# Trigger patterns : veto patterns, a minimal showcase (the 2023 dump)

*This first look uses the February 2023 dump (`data/patterns.txt`, start planes 8-9, earlier RoI code). The main analysis, on the regenerated dump of
all planes with the thesis-era RoI code, measured backsplash and accidental hits, is [`channelBudgetShowcase.ipynb`](channelBudgetShowcase.ipynb);
data/README.md explains the difference.*

The HERD FIT triggers on **patterns**: 3 hits in 3 consecutive same-view planes, at most one mat apart. Each pattern has a **veto region (RoI)** in the PSD:
the tiles a charged particle crossing those mats must have gone through (legacy/aeffcalc/algos/UlegGeomPsdRoi.h). A pattern is vetoed when any tile of its own RoI fires,
not when any tile of the PSD fires, so that backsplash far from the track does not kill good photons.

The trigger electronics cannot carry one line per pattern and one per tile, so **both signals have to be merged**:

| | merged into | hardware |
|---|---|---|
| trigger patterns | $N_G$ pattern groups $g(i)$ | one veto decision per group |
| PSD tiles | $N_C$ tile channels $c(t)$ | tiles OR-ed into one channel |
| routing | $S$, an $N_G\times N_C$ boolean matrix | channel $l$ vetoes group $k$ |

so the mask of 3IR pattern $i$ is $V_{it} = S_{g(i)c(t)}$, to be compared with its RoI $\{T\}_i$, i.e. $M_{it}$ (thesis notation; see channelBudgetShowcase.ipynb, section 1). A tile in $V$ but not in $M$ costs photons (false vetoes);
a tile in $M$ but not in $V$ lets background through. This notebook:

1. reads the pattern $\to$ tile map dumped by ProduceRoIMap in February 2023 (`data/patterns.txt`) into $M$;
2. solves the **0%-loss case** in closed form ($N_G^*$, $N_C^*$) and bounds the fan-out variant (the Boolean rank);
3. shows what a channel budget below $N_G^*, N_C^*$ costs, with a greedy baseline and a geometric placeholder cost, the starting point for the event-weighted optimisation (stochastic/README.md).

In [ ]:
import sys
sys.path.insert(0, "../analytic")

import numpy as np
import matplotlib.pyplot as plt

import patternDump as pd
import losslessUtils as lu
import mergeUtils as mu
import plotUtils as pu

pu.setStyle()

## 1. The dump

426 patterns of the compact-v2 layout, all five FIT sides, starting in planes 8 and 9, with the "Calo" trigger configuration (zig-zag codes 544, 546, 564, 566 removed, a calorimeter
cube required in the RoI). Tiles are keyed (PSD face, PSD layer, volume ID); the PSD has $2\times(14\times14) + 4\times2\times(13\times9) = 1328$ tiles.

In [ ]:
patterns = pd.parsePatternDump(pd.legacyDumpPath)          # The February 2023 dump; the main notebook uses the regenerated one.
incidence = pd.buildIncidence(patterns)
print(f"{incidence.shape[0]} patterns x {incidence.shape[1]} tiles, {incidence.matrix.sum()} RoI entries; tiles in no RoI: {(~incidence.matrix.any(axis=0)).sum()}")
print(f"\n{'FIT side':8} {'patterns':>8} {'planes':>7} {'codes':>22} {'tiles in RoI (mean)':>20} {'tiles used':>11}")
for side in pd.psdSides:
    sub = incidence.forFitSides(side)
    codes = sorted({p.code for p in sub.patterns})
    print(f"{side:8} {sub.shape[0]:8d} {str(sorted({p.firstPlane for p in sub.patterns})):>7} {','.join(codes):>22} {sub.matrix.sum(axis=1).mean():20.1f} {sub.matrix.any(axis=0).sum():11d}")

A vertical pattern of the top FIT, and the same start with a one-mat step: the RoIs are wide (in this view a pattern only constrains one coordinate, so its RoI on the top PSD is a band of whole tile columns) and they overlap heavily,
which is what makes merging possible at all. Shade: number of PSD layers (0, 1, 2) in the RoI at that grid position.

In [ ]:
byName = {p.name: p for p in patterns}
for name in ("TOP806555", "TOP806556"):
    pu.plotRoiOnFaces(byName[name])
plt.show()

How the RoIs are organised. Left: the incidence matrix $M$ with the patterns in geometric order (FIT face, view, first mat, slope) and the tiles grouped by PSD face,
each face's tiles ordered by the patterns that use them. Each FIT face lives mostly on its own PSD face (the diagonal blocks) and reaches a few rows into the
neighbouring faces (the thin strips off the diagonal), never the opposite one. Right: how much the RoIs of the top FIT overlap (Jaccard index
$|\{T\}_a\cap\{T\}_b|/|\{T\}_a\cup\{T\}_b|$). Within a view, RoIs overlap strongly (bands along the diagonal, period 5 = the five slopes per mat); across views they
overlap at about 0.3, because the two views constrain orthogonal coordinates. Strong overlap is what lets patterns share a veto decision.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
blues = LinearSegmentedColormap.from_list("blues", pu.sequentialRamp)
top = incidence.forFitSides("TOP").withoutEmptyColumns()
lossless = lu.findLosslessChannelMap(top.matrix)

figure, axes = plt.subplots(1, 2, figsize=(16, 6.2), constrained_layout=True, gridspec_kw=dict(width_ratios=[1.7, 1]))
pu.plotOrderedIncidence(axes[0], incidence.matrix, patterns, incidence.tileKeys, f"RoI incidence M: {incidence.shape[0]} patterns x {incidence.shape[1]} tiles")
topRows = [i for i in pu.patternOrder(patterns) if patterns[i].fitSide == "TOP"]
image = axes[1].imshow(pu.roiJaccard(incidence.matrix[topRows]), cmap=blues, vmin=0, vmax=1, interpolation="nearest")
evenCount = sum(patterns[i].firstPlane % 2 == 0 for i in topRows)
for drawingAxis in (axes[1].axhline, axes[1].axvline):
    drawingAxis(evenCount - 0.5, color="white", linewidth=1.5)
ticks = [evenCount / 2, evenCount + (len(topRows) - evenCount) / 2]
axes[1].set_xticks(ticks, ["even view\n(plane 8)", "odd view\n(plane 9)"]); axes[1].set_yticks(ticks, ["even view", "odd view"], rotation=90, va="center")
axes[1].set(title="Top FIT: RoI overlap (Jaccard), patterns ordered by mat then slope"); axes[1].grid(False)
figure.colorbar(image, ax=axes[1], shrink=0.85, label="Jaccard index")
plt.show()

The same information per tile, on the unfolded PSD of the thesis (Fig. 4.9: TOP in the centre, each lateral face folded out along its shared edge): how many of the
426 RoIs contain each tile. The edge tiles are the busiest, since the RoIs of two or three FIT faces meet there; they are also the tiles that a per-face channel
layout has to share between faces.

In [ ]:
roisPerTile = dict(zip(incidence.tileKeys, incidence.matrix.sum(axis=0)))
figure, axes = plt.subplots(1, 2, figsize=(13, 6.2), constrained_layout=True)
norm = Normalize(0, incidence.matrix.sum(axis=0).max())
for layer in (0, 1):
    collection = pu.plotUnfoldedPsd(axes[layer], roisPerTile, layer, norm, blues, f"PSD layer {layer}")
figure.colorbar(collection, ax=axes, shrink=0.8, label="RoIs containing the tile")
plt.show()

## 2. The 0%-loss case, analytically

With merges that partition the patterns and the tiles (a signal goes to one channel), $V = M$ exactly requires

- patterns in one group to have **identical RoIs**: $N_G \ge N_G^* = $ number of distinct rows of $M$;
- tiles in one channel to belong to **exactly the same RoIs**: $N_C \ge N_C^* = $ number of distinct columns of $M$;

and grouping identical rows and identical columns reaches both bounds together, so the lossless minimum is $(N_G^*, N_C^*)$, closed form.
If a tile may feed several OR channels (fan-out inside the trigger logic), the fewest channels is the **Boolean rank** of $M$: the minimum number of
bicliques (pattern set $\times$ tile set, all ones) whose union is $M$. That is the biclustering problem proper; it is NP-hard, so it is bounded here:
a fooling set below, a greedy cover and $\min(N_G^*, N_C^*)$ above.

In [ ]:
print(f"{'FIT side':8} {'patterns':>8} {'tiles':>6} {'N_G*':>5} {'N_C*':>5} {'Boolean rank':>14}")
summary = {}
for side in pd.psdSides + ("all",):
    sub = (incidence if side == "all" else incidence.forFitSides(side)).withoutEmptyColumns()
    channelMap = lu.findLosslessChannelMap(sub.matrix)
    assert np.array_equal(channelMap.vetoMask(), sub.matrix)            # Exact: every RoI reproduced.
    lower, upper = lu.computeBooleanRankBounds(sub.matrix)
    summary[side] = (sub.shape, channelMap.routing.shape, (lower, upper))
    print(f"{side:8} {sub.shape[0]:8d} {sub.shape[1]:6d} {channelMap.numberOfGroups:5d} {channelMap.numberOfChannels:5d} {f'{lower} to {upper}':>14}")

Reading the table:

- **Patterns compress well, tiles do not.** About 40% of the patterns share their RoI with another one, but almost every tile has its own set of RoIs, so a
  lossless tile merge saves only a few percent of the channels. Any real channel budget for the tiles is therefore **lossy**.
- **The fan-out model helps on the pattern side.** Over all sides the Boolean rank lies between the fooling-set bound and $N_G^*$: if the trigger logic can send a tile to several OR channels,
  the number of veto lines can drop below the number of distinct RoIs, at no loss. The bounds are loose; closing them (an exact ILP or SAT solve per FIT side) is the first analytic next step.

## 3. Below the lossless counts: what a channel budget costs

With **union routing** (channel $l$ vetoes group $k$ if any of its tiles is in any RoI of the group) every mask contains its RoI: no background leak,
only extra tiles in masks. The cost of a map is $\sum_{i,t} (V - M)_{it}\,C_{it}$ with $C_{it}$ the expected false vetoes of good photons when tile $t$ is added to the mask of pattern $i$.
Until Geant4 events give $C$ (pattern rates $\times$ backsplash probability per tile), $C = 1$: the cost counts the (pattern, tile) pairs added to masks, shown here relative to the
pairs in the RoIs (the **mask excess** $\xi$). The greedy merge below is a baseline for the optimisers to beat.

In [ ]:
channelBudgets = (544, 256, 128, 64, 32, 16)
roiEntries = top.matrix.sum()
curves = {}
for channels in channelBudgets:
    channelMap, history = mu.mergeGreedily(top.matrix, 1, channels, record=True)
    curves[channels] = history
check = mu.mergeGreedily(top.matrix, 16, 64)
addedCost, leaked = mu.computeMaskCost(top.matrix, check)
assert leaked == 0 and np.isclose(addedCost, curves[64][16])           # Union routing never leaks; the incremental cost equals the recomputed one.
print(f"N_G = 16 groups, N_C = 64 channels: mask excess {addedCost / roiEntries:.1%}, leaked RoI entries {leaked}")

In [ ]:
figure, drawingAxis = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
for colour, channels in zip(pu.ordinalRamp, channelBudgets):
    groups = np.array(sorted(curves[channels]))
    inflation = np.array([curves[channels][k] for k in groups]) / roiEntries
    drawingAxis.plot(groups, 100 * inflation, color=colour, label=f"$N_C$ = {channels}" + (" (lossless tiles)" if channels == lossless.numberOfChannels else ""))
drawingAxis.axvline(lossless.numberOfGroups, color=pu.inkMuted, linewidth=1.0, linestyle=":")
drawingAxis.annotate(f"$N_G^*$ = {lossless.numberOfGroups}", (lossless.numberOfGroups, 1), xytext=(-6, 4), textcoords="offset points", ha="right", color=pu.inkMuted)
drawingAxis.set(xscale="log", xlabel="pattern groups $N_G$", ylabel=r"mask excess $\xi$ [%]", yscale="log", ylim=(1, None), xlim=(2, 80), title="Top FIT, greedy union-routed merge (zero leak), placeholder cost C = 1")
drawingAxis.legend(title="PSD channels", loc="lower left")
plt.show()

Reading the curves (log scale): at full tile granularity the pattern groups can drop from $N_G^* = 64$ to 32 for a mask excess of 6.5%, while cutting the tile channels costs
from the start (about 10% at $N_C = 128$ even with $N_G = N_G^*$), consistent with $N_C^* \approx$ the number of tiles. Which of the two budgets binds, and where the knee is, depends on $C$: with real backsplash rates the far, rarely hit
tiles become cheap to merge and the near ones expensive.

## 4. Next steps

- **Analytic, 0% loss:** exact Boolean rank per FIT side (ILP or SAT on the deduplicated matrix, a few hundred columns), and the same with routing constraints of the real trigger board.
- **Event-weighted cost:** fill $C_{it}$ from Geant4 photons (pattern of the first triggered planes $\times$ PSD hits outside the RoI) and measure the leak with protons and electrons, the
  AeffCalc samples or new ones; split into training and test events.
- **Iterative optimisation:** start from the greedy map and improve it on the events: local moves with simulated annealing or basin hopping on the assignments $g, c$; a relaxed, soft
  assignment trained by SGD (sigmoid routing, Gumbel-softmax groups) in JAX; then re-generate, re-weight and iterate. See stochastic/README.md.